### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [ ]:
%%capture
!pip install -U transformers datasets==3.6.0 accelerate evaluate soundfile librosa jiwer

### Import required libraries for pretraining pipeline

In [ ]:
import os
import math
import random
import numpy as np
import torch

from dataclasses import dataclass
from typing import Dict, List, Optional, Union

from datasets import load_dataset, DatasetDict, Audio
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForPreTraining,
    TrainingArguments,
    Trainer,
    set_seed,
)
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    _compute_mask_indices,
    _sample_negative_indices,
)

### Set random seed for reproducibility and print environment info

In [ ]:
set_seed(42)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


### Define experiment configuration: model, dataset, audio parameters, and training hyperparameters

In [ ]:
BASE_MODEL = "facebook/wav2vec2-large-xlsr-53"
DATASET_NAME = "keshan/large-sinhala-asr-dataset"
DATASET_CONFIG = "si"

SAMPLING_RATE = 16_000
MIN_DURATION_SEC = 1.0
MAX_DURATION_SEC = 15.0   # slightly broader than your current 10s cap

TRAIN_SPLIT = "train[:20%]"
VALID_SPLIT = "test[:20%]"   # keep the original test split fully for monitoring

OUTPUT_DIR = "./wav2vec2_sinhala_continual_pretraining"

PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.08
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 50
EVAL_STEPS = 500
SAVE_STEPS = 500

### Install the wget package for downloading files

In [ ]:
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=67408a64de4f9f5dcc243dfec8b04a6a424bb4699295be0c81de0cbef2b15eff
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


### Load the Sinhala dataset into train and validation splits

In [ ]:
train_ds = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=TRAIN_SPLIT,
    trust_remote_code=True,
)

valid_ds = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
    split=VALID_SPLIT,
    trust_remote_code=True,
)

dataset = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
})

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['filename', 'x', 'sentence', 'file'],
        num_rows: 26509
    })
    validation: Dataset({
        features: ['filename', 'x', 'sentence', 'file'],
        num_rows: 4678
    })
})

### Rename the 'file' column to 'audio' and drop unused columns for downstream compatibility

In [ ]:
dataset["train"] = dataset["train"].rename_column("file", "audio")
dataset["validation"] = dataset["validation"].rename_column("file", "audio")

dataset["train"] = dataset["train"].cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
dataset["validation"] = dataset["validation"].cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

### Filter out audio samples outside the allowed duration range

In [ ]:
def keep_audio_range(example):
    audio = example["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    return MIN_DURATION_SEC <= duration <= MAX_DURATION_SEC

dataset["train"] = dataset["train"].filter(keep_audio_range, num_proc=1)
dataset["validation"] = dataset["validation"].filter(keep_audio_range, num_proc=1)

print(dataset)

Filter:   0%|          | 0/26509 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4678 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['filename', 'x', 'sentence', 'audio'],
        num_rows: 26476
    })
    validation: Dataset({
        features: ['filename', 'x', 'sentence', 'audio'],
        num_rows: 4674
    })
})


### Preview a random training sample to verify data loading

In [ ]:
idx = random.randint(0, len(dataset["train"]) - 1)
sample = dataset["train"][idx]

print("Sentence exists but is not used for SSL pretraining:")
print(sample.get("sentence", "N/A"))
print("Audio length:", len(sample["audio"]["array"]))
print("Sampling rate:", sample["audio"]["sampling_rate"])

Sentence exists but is not used for SSL pretraining:
අනිත් පැත්තෙන් ථේරවාදී බුුදුදහමේ
Audio length: 52800
Sampling rate: 16000


### Load the pre-trained feature extractor with attention mask enabled

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)
feature_extractor.return_attention_mask = True

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

### Define the dataset preparation function: extract audio input values and encode text labels, then apply to train and test sets

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    processed = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )

    batch["input_values"] = processed["input_values"][0]
    batch["attention_mask"] = processed["attention_mask"][0]
    batch["input_length"] = len(batch["input_values"])
    return batch

dataset["train"] = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
)

dataset["validation"] = dataset["validation"].map(
    prepare_dataset,
    remove_columns=dataset["validation"].column_names,
    num_proc=1,
)

print(dataset["train"][0].keys())

Map:   0%|          | 0/26476 [00:00<?, ? examples/s]

Map:   0%|          | 0/4674 [00:00<?, ? examples/s]

dict_keys(['input_values', 'attention_mask', 'input_length'])


### Load the XLSR-53 model configured for self-supervised continual pretraining

In [ ]:
model = Wav2Vec2ForPreTraining.from_pretrained(
    BASE_MODEL,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.065,
    mask_time_length=10,
    layerdrop=0.05,
)

Loading weights:   0%|          | 0/429 [00:00<?, ?it/s]

### Freeze the convolutional feature encoder to preserve learned acoustic representations

In [ ]:
model.freeze_feature_encoder()
print("Feature encoder frozen.")

Feature encoder frozen.


### Define a data collator for Wav2Vec2 self-supervised pretraining with masking

In [ ]:
@dataclass
class DataCollatorForWav2Vec2Pretraining:
    model: Wav2Vec2ForPreTraining
    feature_extractor: Wav2Vec2FeatureExtractor
    padding: Union[bool, str] = "longest"
    pad_to_multiple_of: Optional[int] = 8

    def __call__(self, features: List[Dict[str, Union[List[float], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        batch = self.feature_extractor.pad(
            [
                {
                    "input_values": f["input_values"],
                    "attention_mask": f["attention_mask"],
                }
                for f in features
            ],
            padding=self.padding,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch_size = batch["input_values"].shape[0]

        mask_indices_seq_length = int(
            self.model._get_feat_extract_output_lengths(batch["input_values"].shape[-1])
        )

        output_lengths = self.model._get_feat_extract_output_lengths(
            batch["attention_mask"].sum(-1)
        ).to(torch.long)

        sub_attention_mask = torch.zeros(
            (batch_size, mask_indices_seq_length), dtype=torch.long
        )
        sub_attention_mask[(torch.arange(batch_size), output_lengths - 1)] = 1
        sub_attention_mask = sub_attention_mask.flip([-1]).cumsum(-1).flip([-1]).bool()

        mask_time_indices = _compute_mask_indices(
            shape=(batch_size, mask_indices_seq_length),
            mask_prob=self.model.config.mask_time_prob,
            mask_length=self.model.config.mask_time_length,
            attention_mask=sub_attention_mask,
            min_masks=2,
        )

        sampled_negative_indices = _sample_negative_indices(
            features_shape=(batch_size, mask_indices_seq_length),
            num_negatives=self.model.config.num_negatives,
            mask_time_indices=mask_time_indices,
        )

        batch["mask_time_indices"] = torch.tensor(mask_time_indices, dtype=torch.long)
        batch["sampled_negative_indices"] = torch.tensor(sampled_negative_indices, dtype=torch.long)

        return batch

### Define a custom Trainer subclass for Wav2Vec2 pretraining loss computation

In [ ]:
class Wav2Vec2PretrainingTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

### Define a data collator for Wav2Vec2 self-supervised pretraining with masking

In [ ]:
data_collator = DataCollatorForWav2Vec2Pretraining(
    model=model,
    feature_extractor=feature_extractor,
    pad_to_multiple_of=8,
)

### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.08,
    weight_decay=0.01,
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
    save_total_limit=2,
    dataloader_num_workers=2,
    max_grad_norm=1.0,
    prediction_loss_only=True,
    label_names=["mask_time_indices", "sampled_negative_indices"],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### Create the pretraining Trainer instance with model, datasets, and collator

In [ ]:
trainer = Wav2Vec2PretrainingTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)

### Check for existing checkpoints to enable training resumption

In [ ]:
last_checkpoint = None

if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        d for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))
        last_checkpoint = os.path.join(OUTPUT_DIR, checkpoints[-1])

print("Resuming from:", last_checkpoint)

Resuming from: None


### Start or resume the self-supervised pretraining run

In [ ]:
trainer.train(resume_from_checkpoint=last_checkpoint)

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


Step,Training Loss,Validation Loss
500,516.709102,479.142395
1000,518.870820,477.951385
1500,505.989492,475.110931
2000,504.125430,466.789581
2500,494.856562,471.260284
3000,494.609102,468.439331
3500,499.487930,467.357239
4000,483.793086,466.244110
4500,501.867070,469.610229
4965,495.302188,462.954437


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4965, training_loss=509.62339127800857, metrics={'train_runtime': 4345.0003, 'train_samples_per_second': 18.28, 'train_steps_per_second': 1.143, 'total_flos': 1.6479878310140314e+19, 'train_loss': 509.62339127800857, 'epoch': 3.0})

### Save the fine-tuned model and processor to disk

In [ ]:
final_dir = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_dir)
feature_extractor.save_pretrained(final_dir)

print("Saved to:", final_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./wav2vec2_sinhala_continual_pretraining/final


### Mount Google Drive for persistent storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Copy the best checkpoint to Google Drive for persistent storage

In [ ]:
!cp -r ./wav2vec2_sinhala_continual_pretraining /content/drive/MyDrive/

### Check GPU availability and display GPU information

In [ ]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Thu Apr  2 07:42:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [ ]:
%%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Load the Dhivehi (Maldivian) Common Voice dataset (train+validation and test splits)

In [ ]:
from datasets import load_dataset

common_voice_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
common_voice_test = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 115547.86it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 128716.72it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 106006.21it/s]


### Remove unused metadata columns (accent, age, client_id, etc.) from train and test sets and display random samples

In [ ]:
common_voice_train = common_voice_train.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)
common_voice_test = common_voice_test.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)


,sentence,variant
0,ހަމަ ކުޑައިރުކޮޅެއްގެ ފަހުން ފޯނަށް މެސެޖެއް އައިމާ ބަލައިލިއިރު އެވީ އާޒިމާގެ މެސެޖަކަށް,
1,ކޮންމެވެސް ކަހަލަ ބޮލުގެ ރިހުމެއްވެސް ހުރޭ,
2,އަޅުގަނޑުމެންވެސް ފަޅުރަށްރަށަށް ދަތުރުދާ ފަދައިން,
3,އިސާހިތު ދުވަމުން އައިސް ޙުސްނީއާއި ބަދުރިއްޔާ އެކޮޓަރިއަށް ވަދެއްޖެ,
4,ފަހަތުގައި ހުންނަ ކުޑަ ކުނބަށް ވެސް ނަގަނީ ދެރިޔަލު,
5,ރ.ކިނޮޅަހު ޤުރްއާން ކުލާސް މަރާމާތުކުރުން,
6,ބައެއް ޖޯޑުތައް ރާޅު އަރައި ހިސާބުން އެއްގަމުގައި ހިކި ދޮންވެލިގަނޑު މަތީ އިށީނދެލައިގެން ވެއްޔާ ކުޅެލަކުޅެލައި ތިވި,
7,އިންޓަރންޝިޕްގެ ފުރުޞަތު,
8,އޭނާ ލާރި ބޭނުންވެގެން ބުނެފިނަމަ ސުވާލެއް ނެތި އީސާފުޅުބެ ދީއުޅެނީ މިހެންވެގެން ކަންނޭނގެ,
9,އަނެކަކު އާނއެއް,


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [ ]:
import re

# remove punctuation etc. (expanded to include – and ’)
chars_to_remove_regex = r"""[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«\–\’]"""

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, "", batch["sentence"]).lower().strip()
    return batch

common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test  = common_voice_test.map(remove_special_characters)

# remove rows that contain Arabic characters (expanded ranges)
# - Arabic: \u0600-\u06FF
# - Arabic Supplement/Extended and presentation forms commonly used for ligatures:
#   \u0750-\u077F, \u08A0-\u08FF, \uFB50-\uFDFF, \uFE70-\uFEFF
arabic_pattern = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")

def remove_arabic_rows(batch):
    # True if NO Arabic chars → keep row
    return not bool(arabic_pattern.search(batch["sentence"]))

common_voice_train = common_voice_train.filter(remove_arabic_rows)
common_voice_test  = common_voice_test.filter(remove_arabic_rows)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

### Extract the unique character vocabulary from train and test sets

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Remove any remaining Latin characters from transcriptions and re-extract the character vocabulary

In [ ]:
def remove_latin_characters(batch):
    batch["sentence"] = re.sub(r"[a-z]+", "", batch["sentence"])
    return batch

# remove latin characters
common_voice_train = common_voice_train.map(remove_latin_characters)
common_voice_test = common_voice_test.map(remove_latin_characters)

# extract unique characters again
vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [ ]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 51


### Initialize the CTC tokenizer, Wav2Vec2 feature extractor, and Wav2Vec2 processor

In [ ]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [ ]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/e24cb6e0896730e3c68f4266eadc6dbf6150fbe071ba9968bf13481404180b10/dv_train_0/common_voice_dv_18580646.mp3', 'array': array([1.45519152e-11, 1.01863407e-10, 1.30967237e-10, ...,
       7.25846985e-06, 1.29183172e-06, 5.80571304e-06]), 'sampling_rate': 16000}
Target text: ނިރުބަވެރިކަމުގެ ރާޅު އިސާހިތަކު ފަލަސްޠީނުގެ އެހެން ރަށްރަށަށް ވެސް ފެތުރިއްޖެ
Input array shape: (103296,)
Sampling rate: 16000


### Define the dataset preparation function: extract audio input values and encode text labels, then apply to train and test sets

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids

    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels

        return batch

### Define the evaluation metrics function computing WER and CER using greedy decoding

In [ ]:
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the continually pre-trained Wav2Vec2 model with a new CTC head for Dhivehi fine-tuning

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "/content/drive/MyDrive/wav2vec2_sinhala_continual_pretraining/final",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)

### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [ ]:
from transformers import TrainingArguments, Trainer,

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=2700,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor
)


/tmp/ipykernel_816/723348825.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [ ]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 52, 'bos_token_id': 51}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2796.136700,243.184723,0.616390,0.105402


Step,Training Loss,Validation Loss,Wer,Cer
300,2796.136700,243.184723,0.616390,0.105402
600,827.099200,180.488907,0.514697,0.085254
900,643.337800,151.133698,0.457022,0.072180
1200,538.069700,132.426590,0.421318,0.062654


TrainOutput(global_step=1440, training_loss=1074.7853841145834, metrics={'train_runtime': 2942.8489, 'train_samples_per_second': 15.631, 'train_steps_per_second': 0.489, 'total_flos': 7.110251700446906e+18, 'train_loss': 1074.7853841145834, 'epoch': 10.0})

### Start the fine-tuning training run

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
300,494.121400,149.056290,0.423990,0.063876
600,620.334800,157.480240,0.449673,0.069736


Step,Training Loss,Validation Loss,Wer,Cer
300,494.121400,149.056290,0.423990,0.063876
600,620.334800,157.480240,0.449673,0.069736
900,478.518800,143.101349,0.424510,0.063884
1200,426.406200,137.553558,0.412856,0.061291


TrainOutput(global_step=1440, training_loss=482.53397352430557, metrics={'train_runtime': 2511.2811, 'train_samples_per_second': 18.317, 'train_steps_per_second': 0.573, 'total_flos': 7.109995886306734e+18, 'train_loss': 482.53397352430557, 'epoch': 10.0})

### Run greedy CTC inference on the test set and compute WER/CER

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
import numpy as np

all_pred = []
all_refs = []

checkpoint_dir = "./outputs/checkpoint-1440"

model = Wav2Vec2ForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2Processor.from_pretrained(checkpoint_dir)

model.eval()

for ex in common_voice_test:
    input_values = torch.tensor(ex["input_values"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0].lower()
    all_pred.append(pred_text)

    label_ids = ex["labels"]
    label_ids = [id_ for id_ in label_ids if id_ != -100]

    ref_text = processor.decode(label_ids, group_tokens=False).lower()
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (ASR only, greedy): {wer:.4f}")
print(f"CER (ASR only, greedy): {cer:.4f}")

for idx in range(3):
    print(f"\n--- Example {idx} ---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (ASR only, greedy): 0.4054
CER (ASR only, greedy): 0.0588

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
PRED: ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
PRED: އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
PRED: އެސްފިޔަތަކުން ވަނީ ނިދިގެއްލިފަ


### Mount Google Drive for persistent storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Copy the best checkpoint to Google Drive for persistent storage

In [ ]:
!cp -r ./outputs/checkpoint-1440 /content/drive/MyDrive/final_model65